# Sequence reconstruction fidelity - Canonical Jaccard Index

Reproduces **Tables 6 and 7** of the paper.

**What is measured.** Given a pair `(reference, suspect)` the model predicts a
transformation sequence; we score it against the ground truth with the
**Canonical Jaccard Index**, which is invariant to the equivalences described in
the paper (Section 5.1):

1. *Geometric canonicalisation* - rotations and flips form the dihedral group
   `D_4`. The geometric sub-sequence is collapsed to a single canonical token by
   multiplying the corresponding 2x2 matrices
   (`seq_iou_bench.canonicalize_sequence`); e.g. `(horizontal_flip, vertical_flip)`
   and `(rotate_180)` map to the same token. An identity result is dropped.
2. *Commutative set* - the photometric / noise / structural ops
   (`grayscale`, `color_jitter`, `noise_adding`, `jpeg_artefacts`, `crop`) are
   order-independent and treated as an unordered set.
3. The two canonical sets are compared with a micro Jaccard / IoU:
   `sum |C(pred) & C(gt)| / sum |C(pred) | C(gt)|`.

Two protocols, as in the paper:
- **DomainNet** (Table 6): one random sequence per test image - `DomainNetFullBenchmark`.
- **Curated negatives** (Table 7): 1000 samples per sequence length 1..5 -
  `LengthWiseAccuracyBenchmark` (and `QwenLengthWiseBenchmark` for the VLM baseline).

Set the checkpoints/dataset roots in the next cell. The curated negative dataset
is **not public** (see the paper, *Data availability*); point `NEGATIVE_ROOT` /
`NEGATIVE_JSON` at your own negatives to reproduce Table 7.


In [ ]:
# ============================================================
# Paths - edit these to point at YOUR local files.
# Nothing below is tied to a specific machine.
# ============================================================
import os

# Model configs shipped with the repo:
CONFIG_EFFNET = "../configs/train_config_effnet.yaml"
CONFIG_VIT    = "../configs/train_config_vit.yaml"

# Model weights: point these at the checkpoints YOU trained.
#   *_PRE : after self-supervised pre-training -> scripts/run_train.py  / run_train_siamnet.py
#   *_SFT : after supervised fine-tuning       -> scripts/run_tune.py   / run_tune_siamnet.py
# Each file is a training checkpoint containing a "model_state_dict" entry.
EFFNET_PRE  = "PATH/TO/effnet_pretrain/checkpoint_epoch_XX.pth"
EFFNET_SFT  = "PATH/TO/effnet_tune/checkpoint_epoch_XX.pth"
VIT_PRE     = "PATH/TO/vit_pretrain/checkpoint_epoch_XX.pth"
VIT_SFT     = "PATH/TO/vit_tune/checkpoint_epoch_XX.pth"
SIAMNET_PRE = "PATH/TO/siamnet_pretrain/checkpoint_epoch_XX.pth"
SIAMNET_SFT = "PATH/TO/siamnet_tune/checkpoint_epoch_XX.pth"

# Datasets:
DOMAINNET_ROOT = "PATH/TO/domainnet"       # public: http://ai.bu.edu/M3SDA/
NEGATIVE_ROOT  = "PATH/TO/negative_pairs"  # curated negatives - NOT public (see paper, Data availability)
NEGATIVE_JSON  = "filtered_negative_test_dataset_meta.json"  # produced by filter_good_pairs.py

# External evaluators / dataset for the curated negative set live in the authors'
# dataset-generation repo (pairwise_comparison_validation, categorization_visualization).
DATASET_GEN_SRC = "PATH/TO/dataset-generation/src"

# Where benchmark JSON / figures are written:
OUTPUT_DIR = "metrics"


In [ ]:
import pandas as pd
import math
import matplotlib.pyplot as plt
import json
from pathlib import Path
import random
from PIL import Image
import os
import sys
import io
import numpy as np
import torch
from omegaconf import OmegaConf
from torchvision import transforms

src_path = os.path.join(os.path.abspath('..'), 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from seq_iou_bench import LengthWiseAccuracyBenchmark
from dataset import ImageTransformer
from dataset import TransformTokenizer
from model import ImageTransformPredictor

%load_ext autoreload
%autoreload 2

## DomainNet

In [ ]:
from seq_iou_bench import DomainNetFullBenchmark

In [ ]:
preprocess_effnet = transforms.Compose([
            transforms.Resize((300, 300)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

config = OmegaConf.load(CONFIG_EFFNET)
model_effnet = ImageTransformPredictor(config.model)

checkpoint_path = EFFNET_PRE
# checkpoint_path = EFFNET_SFT

checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
model_effnet.load_state_dict(checkpoint['model_state_dict'])
model_effnet.to('cuda')
model_effnet.eval()
print('ready')

In [ ]:
preprocess_vit = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

config = OmegaConf.load(CONFIG_VIT)
model_vit = ImageTransformPredictor(config.model)

checkpoint_path = VIT_PRE
# checkpoint_path = VIT_SFT

checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
model_vit.load_state_dict(checkpoint['model_state_dict'])
model_vit.to('cuda')
model_vit.eval()
print('ready')

In [ ]:
import torch

from transformers import AutoTokenizer, AutoProcessor
from transformers import Qwen3VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
from PIL import Image

if not hasattr(torch.compiler, "is_compiling"):
    torch.compiler.is_compiling = lambda: False

model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-4B-Instruct",
)

# Store processor for preprocessing
processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-4B-Instruct")

model.to("cuda")
print('ready')

In [ ]:
total = model.num_parameters()                       # = sum(p.numel() ...), tied-веса считаются один раз
print(f"{total:,}  ({total/1e9:.3f} B)")

In [ ]:
tokenizer = TransformTokenizer()
transformer = ImageTransformer()

In [ ]:
data_path = DOMAINNET_ROOT

In [ ]:
device = torch.device("cuda")

benchmark = DomainNetFullBenchmark(
    model=model_effnet,
    dataset_root=data_path,
    transformer=transformer,
    tokenizer=tokenizer,
    preprocess=preprocess_effnet,
    device=device,
    model_type="custom"
)

results = benchmark.run(model_name="Ours-EffNet-pre", output_path=OUTPUT_DIR + "/seq_iou_domainnet.json")

In [ ]:
device = torch.device("cuda")

benchmark = DomainNetFullBenchmark(
    model=model_vit,
    dataset_root=data_path,
    transformer=transformer,
    tokenizer=tokenizer,
    preprocess=preprocess_vit,
    device=device,
    model_type="custom"
)

results = benchmark.run(model_name="Ours-ViT-pre", output_path=OUTPUT_DIR + "/seq_iou_domainnet.json")

In [ ]:
# device = torch.device("cuda")

# benchmark = DomainNetFullBenchmark(
#     model=model,
#     dataset_root=data_path,
#     transformer=transformer,
#     tokenizer=None,
#     preprocess=processor,
#     device=device,
#     model_type="qwen"
# )

# results = benchmark.run(model_name="Qwen3-VL-4B", output_path=OUTPUT_DIR + "/seq_iou_domainnet.json")

## Negative Dataset

### Evaluation of sequence prediction models (ViT based, EfficientNet based)

In [ ]:
data_json_path = NEGATIVE_JSON

tokenizer = TransformTokenizer()
transformer = ImageTransformer()

In [ ]:
preprocess_effnet = transforms.Compose([
            transforms.Resize((300, 300)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

config = OmegaConf.load(CONFIG_EFFNET)
model_effnet = ImageTransformPredictor(config.model)

checkpoint_path = EFFNET_PRE
# checkpoint_path = EFFNET_SFT

checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
model_effnet.load_state_dict(checkpoint['model_state_dict'])
model_effnet.to('cuda')
model_effnet.eval()
print('ready')

In [ ]:
preprocess_vit = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

config = OmegaConf.load(CONFIG_VIT)
model_vit = ImageTransformPredictor(config.model)

checkpoint_path = VIT_PRE
# checkpoint_path = VIT_SFT

checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
model_vit.load_state_dict(checkpoint['model_state_dict'])
model_vit.to('cuda')
model_vit.eval()
print('ready')

In [ ]:
device = torch.device("cuda")

benchmark = LengthWiseAccuracyBenchmark(
    model=model_vit,
    dataset_root=NEGATIVE_ROOT,
    json_path=data_json_path,
    preprocess=preprocess_vit,
    transformer=transformer,
    tokenizer=tokenizer,
    config=config,
    device=device,
    n_samples_per_length=1000,
    seed=2026
)

results = benchmark.run(model_name="Ours-ViT-sft", output_path=OUTPUT_DIR + "/seq_iou_negative_dataset.json")

In [ ]:
device = torch.device("cuda")

benchmark = LengthWiseAccuracyBenchmark(
    model=model_effnet,
    dataset_root=NEGATIVE_ROOT,
    json_path=data_json_path,
    preprocess=preprocess_effnet,
    transformer=transformer,
    tokenizer=tokenizer,
    config=config,
    device=device,
    n_samples_per_length=1000,
    seed=2026
)

results = benchmark.run(model_name="Ours-EffNet-sft", output_path=OUTPUT_DIR + "/seq_iou_negative_dataset.json")

In [ ]:
device = torch.device("cuda")

benchmark = LengthWiseAccuracyBenchmark(
    model=model_vit,
    dataset_root=NEGATIVE_ROOT,
    json_path=data_json_path,
    preprocess=preprocess_vit,
    transformer=transformer,
    tokenizer=tokenizer,
    config=config,
    device=device,
    n_samples_per_length=1000,
    seed=2026
)

results = benchmark.run(model_name="Ours-ViT-pre", output_path=OUTPUT_DIR + "/seq_iou_negative_dataset.json")

In [ ]:
device = torch.device("cuda")

benchmark = LengthWiseAccuracyBenchmark(
    model=model_effnet,
    dataset_root=NEGATIVE_ROOT,
    json_path=data_json_path,
    preprocess=preprocess_effnet,
    transformer=transformer,
    tokenizer=tokenizer,
    config=config,
    device=device,
    n_samples_per_length=1000,
    seed=2026
)

results = benchmark.run(model_name="Ours-EffNet-pre", output_path=OUTPUT_DIR + "/seq_iou_negative_dataset.json")

### Comparison with Qwen3-VL-4B-Instruct in the task of predicting the transformation sequence

In [ ]:
import torch

from transformers import AutoTokenizer, AutoProcessor
from transformers import Qwen3VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
from PIL import Image

if not hasattr(torch.compiler, "is_compiling"):
    torch.compiler.is_compiling = lambda: False

from seq_iou_bench import QwenLengthWiseBenchmark

In [ ]:
model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-4B-Instruct",
)

# Store processor for preprocessing
processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-4B-Instruct")

#### An example of how we use VLM to predict a sequence

In [ ]:
prompt_seq_prediction = """You are given two images: Image A (original) and Image B (transformed).  
Your task is to predict the sequence of transformations applied to Image A to obtain Image B, using only the following allowed operations:  
"noop", "grayscale", "rotate_90", "rotate_180", "rotate_270", "color_jitter", "noise_adding", "jpeg_artefacts", "crop", "horizontal_flip", "vertical_flip".

- The sequence may contain zero, one, or multiple transformations applied in order.  
- If Image A and Image B are identical, return: ["noop"]
- If Image B can be obtained by applying a sequence of the allowed transformations (in the correct order), return that sequence as a JSON list, e.g.: ["color_jitter", "noise_adding", "rotate_270", "horizontal_flip"]  
- If the transformation from Image A to Image B requires any operation not in the allowed list (e.g., blur, resize, perspective distortion, custom warping, etc.), or if the images are unrelated, return an empty list: []  

Output only the JSON list. Do not add explanations, comments, or extra text."""

In [ ]:
prompt_class_prediction = """You are an expert in image forensics. Determine whether Image B is a plagiarized version of Image A, meaning it was obtained by applying only the following allowed transformations to Image A:  
- Geometric: rotate_90, rotate_180, rotate_270, horizontal_flip, vertical_flip  
- Photometric: grayscale, color_jitter  
- Structural: crop (central square)  
- Noise/compression: noise_adding, jpeg_artefacts  

If Image B can be produced from Image A using any combination of these operations (in any order, without repetition), output 1.  
If Image B involves any other transformation (e.g., blur, resize, perspective warp, object removal, or unrelated content), output 0.

Output only a single digit: 1 or 0. Do not add any other text, explanation, or punctuation."""

In [ ]:
image = Image.open('../assets/dog.jpg')
transformed_image, sequence = transformer.transform_by_length(image, 2)

sequence

In [ ]:
transformed_image

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "image", "image": transformed_image},
            {"type": "text", "text": prompt_class_prediction},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cpu")

# Inference
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text[0])

#### Run benchmark

In [ ]:
from seq_iou_bench import QwenLengthWiseBenchmark

benchmark = QwenLengthWiseBenchmark(
    model=model,
    processor=processor,
    dataset_root=NEGATIVE_ROOT,
    json_path=data_json_path,
    transformer=transformer,
    n_samples_per_length=1000,
    seed=2026
)

benchmark.run(model_name="Qwen3-VL-4B-Instruct", output_path=OUTPUT_DIR + "/seq_iou_negative_dataset.json")